# WP3A — Frozen CLIP compositional zero-shot reference

Co-owners: **Yuen Kai CHIA and Andy Wei Jian CHU**. This notebook is the visible executable workflow; it does not import project ML scripts. Run with the **as04_yolo_katana** WSL kernel. No training is implemented. All model loading, downloads, image decoding and full development inference are opt-in.

## 1. Objective and scope

WP2 learns crop and disease heads from fitting images; this reference instead matches frozen CLIP image embeddings to label-only candidate texts. Compositional OOD means a crop–disease pair is absent from fitting while its components remain present. Evaluate seen and internally unseen folds separately in the same **323-candidate** universe. This is a development reference, not a justified trainable improvement by itself.

Official assignment sources require independent and multi-task baselines, a justified proposed method, Top-1/Top-5 and seen/unseen harmonic mean. The seed-42 development split, fixed prompts, CLIP implementation and exact tie policy below are team engineering choices. Official tests remain reserved for a later frozen evaluation.

No local or published WP3 implementation was found during the audit; coordinate any unpublished Andy work before parallel edits. Jonathan's WP5 PR #1 remains separate. Notebook-local metrics independently represent the agreed candidate/metric contract and do not replace his PR.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from numbers import Integral, Real
from typing import Any
import hashlib
import importlib.metadata as metadata
import json
import math
import platform
import random
import re
import subprocess
import sys
import time
import unicodedata
import uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "DPD_Small_train.csv").is_file() and (directory / "notebooks").is_dir():
            return directory
    raise FileNotFoundError("Open this notebook inside the WSL repository containing DPD_Small_train.csv.")

ROOT = find_project_root()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ENVIRONMENT = {
    "python": sys.version, "torch": torch.__version__, "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    "working_directory": str(Path.cwd()), "project_root": str(ROOT),
}
print(json.dumps(ENVIRONMENT, indent=2))


## 2. Explicit execution controls and fixed experiment policy

**Run All with these defaults does not load a model, decode images, download weights or evaluate folds.** Metadata checks are small and permitted. Full inference requires deliberate enabling of its flag after reviewing the printed configuration. Do not change prompts after seeing results and continue calling the run a pre-specified baseline. The owner will execute B/32 and L/14 as two separately recorded frozen-reference evaluations using the same final v2 policy. B/32 remains the primary reference; this correction does not select a model using development accuracy.


In [ ]:
ACQUIRE_PRETRAINED_WEIGHTS = False
LOAD_CLIP_MODEL = False
RUN_TEXT_ENCODING = False
RUN_IMAGE_SMOKE = False
RUN_DEVELOPMENT_EVALUATION = False
SHOW_IMAGES = False
READ_WP2_RESULTS = False
SAVE_RESULTS = False
RUN_TRAINING = False  # WP3A contains no training; leave False.

CLIP_REVISION = "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"
MODEL_REGISTRY = {
    "ViT-B/32": {
        "filename": "ViT-B-32.pt", "sha256": "40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af",
        "embedding_dimension": 512, "input_resolution": 224,
    },
    "ViT-L/14": {
        "filename": "ViT-L-14.pt", "sha256": "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836",
        "embedding_dimension": 768, "input_resolution": 224,
    },
}
PROMPT_POLICY = {
    "version": "label_only_single_template_v2",
    "normalization": "Unicode NFKC; underscores to spaces; collapse whitespace; lowercase; no aliases/symptoms",
    "healthy": "a photo of a healthy {crop} plant",
    "diseased": "a photo of {article} {crop} plant affected by {disease}",
    "article_rule": "an for normalized crop labels starting with a/e/i/o/u; otherwise a; diseased prompts only; verified for the current 55 crop labels",
}
CONFIG = {
    "model": "ViT-L/14", "seed": 42, "batch_size": 8, "text_batch_size": 32,
    "num_workers": 0, "device": str(DEVICE), "smoke_images": 4,
    "weight_directory": "/home/user1/COS30082_pretrained_weights/clip",
    "output_directory": "/home/user1/COS30082_experiments/wp3_zero_shot",
    "wp2_diagnostic_json": "/home/user1/COS30082_experiments/wp2_m5e_pilot/m5e_pair_20261030T180618Z_bdca43b9/20261030T180620Z_multitask_convnext_base_20131b22/diagnostics_epoch_0001_55c0f74f.json",
    "wp2_run_summary": "/home/user1/COS30082_experiments/wp2/20261030T103349Z_multitask_convnext_base_22e3ef2f/run_summary.json",
    "wp2_pair_run_summary": "/home/user1/COS30082_experiments/wp2_m5e_pilot/m5e_pair_20261030T180618Z_bdca43b9/20261030T180620Z_multitask_convnext_base_20131b22/run_summary.json",
}
def validate_config(config):
    if config["model"] not in MODEL_REGISTRY:
        raise ValueError("Only the pinned B/32 and optional L/14 assets are supported.")
    if type(config["seed"]) is not int or not 0 <= config["seed"] < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    for key in ("batch_size", "text_batch_size", "smoke_images"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    if config["smoke_images"] > 4:
        raise ValueError("Bounded smoke is capped at four fitting images.")
    if type(config["num_workers"]) is not int or config["num_workers"] < 0:
        raise ValueError("num_workers must be nonnegative.")
    if torch.device(config["device"]).type not in ("cpu", "cuda"):
        raise ValueError("Only CPU/CUDA devices are supported.")
    for key in ("weight_directory", "output_directory"):
        if not Path(config[key]).is_absolute():
            raise ValueError(f"{key} must be an absolute path.")
    return dict(config)

validate_config(CONFIG)
if RUN_TRAINING:
    raise RuntimeError("Training is outside WP3A; no trainable loop is implemented.")
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.backends.cudnn.benchmark = False
print(json.dumps({"config": CONFIG, "prompt_policy": PROMPT_POLICY}, indent=2))


## 3. Training-source metadata loading and exact development split

The following code visibly reproduces the accepted two-step algorithm: hold out 20% of compositions while retaining every component in fitting, then classwise 20% seen-image validation with singletons retained in fitting. Global composition IDs are never renumbered. The split seed is **42**, independent of a future experiment seed. Read only the original training CSV. Full file hashing is metadata-only; no image is decoded here.


In [ ]:
COLUMN_NAMES = ["image_name", "crop_id", "disease_id", "composition_id", "crop_name", "disease_name", "composition_name"]
SPLIT_VERSION = "pytorch-v1"
DEFAULT_CSVS = (str(ROOT / "DPD_Small_train.csv"),)
def load_metadata(path):
    frame = pd.read_csv(path, header=None)
    if frame.shape[1] != len(COLUMN_NAMES) or frame.empty:
        raise ValueError(f"Expected non-empty seven-column metadata: {path}")
    frame.columns = COLUMN_NAMES
    if frame.isna().any().any():
        raise ValueError(f"Missing metadata values: {path}")
    for col in ("crop_id", "disease_id", "composition_id"):
        numeric = pd.to_numeric(frame[col], errors="raise")
        if ((numeric < 0) | (numeric % 1 != 0)).any():
            raise ValueError(f"Invalid {col}: {path}")
        frame[col] = numeric.astype("int64")
    for name in frame.image_name:
        if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
            raise ValueError(f"Expected flat image filename, got {name!r}")
    return frame

def validate_metadata(train, seen, unseen):
    frames = [train, seen, unseen]
    for frame in frames:
        if frame.image_name.duplicated().any():
            raise ValueError("Duplicate image names within a split")
    for i in range(3):
        for j in range(i + 1, 3):
            if set(frames[i].image_name) & set(frames[j].image_name):
                raise ValueError("Image filename overlap across official splits")
    all_rows = pd.concat(frames)
    for kind in ("crop", "disease", "composition"):
        pairs = all_rows[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        if pairs[f"{kind}_id"].duplicated().any() or pairs[f"{kind}_name"].duplicated().any():
            raise ValueError(f"Inconsistent {kind} ID/name mapping")
    pairs = all_rows[["composition_id", "crop_id", "disease_id"]].drop_duplicates()
    if pairs.composition_id.duplicated().any() or pairs.duplicated(["crop_id", "disease_id"]).any():
        raise ValueError("Inconsistent composition/component mapping")
    if set(train.composition_id) & set(unseen.composition_id):
        raise ValueError("Official unseen compositions appear in training")
    if not set(seen.composition_id) <= set(train.composition_id):
        raise ValueError("Seen-test contains compositions absent from training")
    for col in ("crop_id", "disease_id"):
        if set(train[col]) != set(range(train[col].nunique())):
            raise ValueError(f"Training {col} must be contiguous and start at zero")
        if not set(all_rows[col]) <= set(train[col]):
            raise ValueError(f"Test {col} absent from training")
    return {"split_sizes": [len(f) for f in frames],
            "num_crops": int(train.crop_id.nunique()),
            "num_diseases": int(train.disease_id.nunique()),
            "num_seen_compositions": int(train.composition_id.nunique()),
            "num_unseen_compositions": int(unseen.composition_id.nunique()),
            "singleton_compositions": int((train.composition_id.value_counts() == 1).sum()),
            "filename_overlap": False, "seen_unseen_composition_overlap": False}

def make_validation_split(frame, val_split=0.2, seed=42, mode="stratified"):
    """Classwise split keeps singletons in train; heldout removes whole classes.

    Classwise rounding makes the realized image fraction approximate.
    heldout mode interprets val_split as a fraction of composition classes.
    """
    if not 0 < val_split < 1:
        raise ValueError("val_split must be between 0 and 1")
    frame = frame.reset_index(drop=True)
    rng = random.Random(seed)
    groups = {int(key): list(group.index) for key, group in frame.groupby("composition_id", sort=True)}
    val_indices = []
    if mode == "stratified":
        for indices in groups.values():
            rng.shuffle(indices)
            n = min(len(indices) - 1, max(1, round(len(indices) * val_split))) if len(indices) > 1 else 0
            val_indices.extend(indices[:n])
    elif mode == "heldout_composition":
        candidates = list(groups)
        rng.shuffle(candidates)
        remaining_crop = Counter(frame.crop_id)
        remaining_disease = Counter(frame.disease_id)
        target = max(1, round(len(groups) * val_split))
        heldout = []
        for comp_id in candidates:
            group = frame.loc[groups[comp_id]]
            crops, diseases = Counter(group.crop_id), Counter(group.disease_id)
            if all(remaining_crop[k] > n for k, n in crops.items()) and all(remaining_disease[k] > n for k, n in diseases.items()):
                heldout.append(comp_id)
                val_indices.extend(groups[comp_id])
                remaining_crop.subtract(crops)
                remaining_disease.subtract(diseases)
                if len(heldout) == target:
                    break
        if len(heldout) < target:
            raise ValueError(f"Only {len(heldout)} valid heldout classes; requested {target}. Reduce val_split.")
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    validation = frame.loc[sorted(val_indices)].reset_index(drop=True)
    training = frame.drop(index=val_indices).reset_index(drop=True)
    if training.empty or validation.empty:
        raise ValueError("Split produced an empty training or validation set")
    for col in ("crop_id", "disease_id"):
        if set(frame[col]) != set(training[col]):
            raise ValueError(f"Split removed a training component: {col}")
    return training, validation

@dataclass(frozen=True)
class DevelopmentSplit:
    """Original labels and one candidate universe for both validation folds.

    Treat the contained DataFrames/mappings as read-only. Frozen field bindings
    do not make pandas objects immutable; DPDataset copies each frame.
    """

    fit: pd.DataFrame
    val_seen: pd.DataFrame
    val_unseen: pd.DataFrame
    mappings: dict[str, Any]
    candidate_ids: tuple[int, ...]
    candidate_pairs: tuple[tuple[int, int], ...]
    source_csv: str
    source_sha256: str
    candidate_sha256: str
    seed: int
    heldout_fraction: float
    seen_fraction: float
    split_version: str = f"{SPLIT_VERSION}/three-way-v1"

def _fraction(value: float, name: str) -> float:
    if isinstance(value, bool) or not isinstance(value, Real) or not 0 < value < 1:
        raise ValueError(f"{name} must be a finite fraction between zero and one")
    return float(value)

def _training_mappings(frame: pd.DataFrame) -> dict[str, Any]:
    """Build only training-derived maps; the combined train/test wrapper is unsuitable."""
    mappings = {}
    for kind in ("crop", "disease"):
        pairs = frame[[f"{kind}_id", f"{kind}_name"]].drop_duplicates()
        pairs = pairs.sort_values(f"{kind}_id")
        id_to_name = {int(key): str(name) for key, name in pairs.itertuples(index=False, name=None)}
        mappings[f"{kind}_id_to_name"] = id_to_name
        mappings[f"{kind}_name_to_id"] = {name: key for key, name in id_to_name.items()}
        mappings[f"num_{kind}s"] = len(id_to_name)
    pairs = frame[["composition_id", "crop_id", "disease_id", "composition_name"]]
    pairs = pairs.drop_duplicates().sort_values("composition_id")
    mappings["composition_id_to_pair"] = {
        int(row.composition_id): (int(row.crop_id), int(row.disease_id))
        for row in pairs.itertuples()
    }
    mappings["composition_id_to_name"] = {
        int(row.composition_id): str(row.composition_name) for row in pairs.itertuples()
    }
    ids = tuple(mappings["composition_id_to_pair"])
    mappings["composition_id_to_index"] = {comp: index for index, comp in enumerate(ids)}
    mappings["composition_index_to_id"] = dict(enumerate(ids))
    mappings["num_compositions"] = len(ids)
    return mappings

def _check_partitions(
    source: pd.DataFrame, fit: pd.DataFrame, seen: pd.DataFrame, unseen: pd.DataFrame,
) -> None:
    combined = pd.concat([fit, seen, unseen], ignore_index=True)
    if (combined.image_name.duplicated().any()
            or set(combined.image_name) != set(source.image_name)):
        raise ValueError("Development partitions must be disjoint and cover training filenames")
    if not source.sort_values("image_name").reset_index(drop=True).equals(
        combined.sort_values("image_name").reset_index(drop=True)
    ):
        raise ValueError("Development partitions changed original metadata rows or label types")
    fit_ids, seen_ids, unseen_ids = (set(frame.composition_id) for frame in (fit, seen, unseen))
    if not seen_ids <= fit_ids or unseen_ids & (fit_ids | seen_ids):
        raise ValueError("Seen coverage or held-out composition isolation failed")
    for column in ("crop_id", "disease_id"):
        if set(fit[column]) != set(source[column]):
            raise ValueError(f"Fitting partition lost training components: {column}")

def build_development_split(
    train_csv: str | Path = DEFAULT_CSVS[0],
    *,
    seed: int = 42,
    heldout_fraction: float = 0.2,
    seen_fraction: float = 0.2,
) -> DevelopmentSplit:
    """Compose existing train-only splits without reading images or writing files.

    The default DPD-Small source yields 4868 fit / 1238 seen / 1894 unseen
    images. heldout_fraction counts compositions; seen_fraction is classwise
    image sampling within the retained portion. Synthetic training CSVs are
    supported for tests. Known official test CSV paths are rejected.
    """
    if isinstance(seed, bool) or not isinstance(seed, Integral) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    seed = int(seed)
    heldout_fraction = _fraction(heldout_fraction, "heldout_fraction")
    seen_fraction = _fraction(seen_fraction, "seen_fraction")
    path = Path(train_csv)
    if path.name != "DPD_Small_train.csv" or path.resolve().name != "DPD_Small_train.csv":
        raise ValueError("Official test metadata cannot be used as development training data")
    before_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    source = load_metadata(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != before_hash:
        raise ValueError("Training CSV changed while loading")
    # Empty control frames reuse WP1 validation without loading either test CSV.
    empty = source.iloc[:0]
    validate_metadata(source, empty, empty)
    retained, unseen = make_validation_split(
        source, val_split=heldout_fraction, seed=seed, mode="heldout_composition"
    )
    fit, seen = make_validation_split(
        retained, val_split=seen_fraction, seed=seed, mode="stratified"
    )
    _check_partitions(source, fit, seen, unseen)
    mappings = _training_mappings(source)
    ids = tuple(mappings["composition_id_to_pair"])
    pairs = tuple(mappings["composition_id_to_pair"][comp] for comp in ids)
    records = [[comp, *pair] for comp, pair in zip(ids, pairs)]
    candidate_hash = hashlib.sha256(
        json.dumps(records, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return DevelopmentSplit(
        fit=fit, val_seen=seen, val_unseen=unseen, mappings=mappings,
        candidate_ids=ids, candidate_pairs=pairs, source_csv=str(path.resolve()),
        source_sha256=before_hash, candidate_sha256=candidate_hash, seed=seed,
        heldout_fraction=heldout_fraction, seen_fraction=seen_fraction,
    )

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def split_identity(split):
    records = {
        name: [[str(row.image_name), int(row.crop_id), int(row.disease_id), int(row.composition_id)]
               for row in frame.itertuples()]
        for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))
    }
    return {"source_sha256": split.source_sha256, "candidate_sha256": split.candidate_sha256,
            "partition_sha256": {name: json_hash(rows) for name, rows in records.items()},
            "split_sha256": json_hash({name: frame.image_name.tolist() for name, frame in (("fit", split.fit), ("val_seen", split.val_seen), ("val_unseen", split.val_unseen))}), "split_version": split.split_version, "split_seed": split.seed}

split = build_development_split(ROOT / "DPD_Small_train.csv", seed=42)
assert (len(split.fit), len(split.val_seen), len(split.val_unseen)) == (4868, 1238, 1894)
assert (split.mappings["num_crops"], split.mappings["num_diseases"], len(split.candidate_ids)) == (55, 175, 323)
SPLIT_IDENTITY = split_identity(split)
display(pd.DataFrame([
    {"fold": name, "images": len(frame), "crops": frame.crop_id.nunique(),
     "diseases": frame.disease_id.nunique(), "compositions": frame.composition_id.nunique()}
    for name, frame in (("fitting", split.fit), ("seen validation", split.val_seen), ("internally unseen", split.val_unseen))
]))
print(json.dumps(SPLIT_IDENTITY, indent=2))
print("Filenames are disjoint; all 8000 rows covered; fitting retains every crop and disease.")


## 4. Fixed candidate prompts and explicit column provenance

Only training-source names define vocabulary. Underscores become spaces, Unicode is normalized, whitespace is collapsed and case is lowered. We do not add symptoms, taxonomy synonyms, prompt ensembles or development-selected descriptions. “Healthy” uses its own grammatical template. Distinct global IDs must not collapse to one normalized prompt. Review this table **before** full development inference and retain its digest.

The candidate universe includes all 323 original training compositions, including the 65 held out from fitting. This is legitimate zero-shot vocabulary, not use of their images for training.

**Pre-evaluation grammar correction — policy v2:** diseased prompts use `an` before normalized crop labels beginning with a/e/i/o/u, otherwise `a`. The current vocabulary has four vowel-initial crops: apple, apricot, eggplant and olive. This changes23 diseased prompts only; all37 healthy prompts retain `a healthy`. Crop/disease names and candidate ordering are unchanged. The policy is fixed before either full development evaluation.


In [ ]:
def normalize_label(label):
    if not isinstance(label, str):
        raise TypeError("Labels must be strings.")
    cleaned = " ".join(unicodedata.normalize("NFKC", label).replace("_", " ").split()).lower()
    if not cleaned:
        raise ValueError("An empty label cannot form a prompt.")
    return cleaned

def crop_indefinite_article(crop_label):
    """Label-only rule, checked against the current 55-crop vocabulary."""
    crop = normalize_label(crop_label)
    return "an" if crop[0] in "aeiou" else "a"

def build_prompt_table(split):
    rows = []
    for column, (global_id, pair) in enumerate(zip(split.candidate_ids, split.candidate_pairs)):
        crop_id, disease_id = pair
        crop = normalize_label(split.mappings["crop_id_to_name"][crop_id])
        disease = normalize_label(split.mappings["disease_id_to_name"][disease_id])
        prompt = (PROMPT_POLICY["healthy"].format(crop=crop) if disease == "healthy"
                  else PROMPT_POLICY["diseased"].format(article=crop_indefinite_article(crop), crop=crop, disease=disease))
        rows.append({"score_column": column, "global_composition_id": global_id,
                     "crop_id": crop_id, "disease_id": disease_id, "crop": crop, "disease": disease,
                     "prompt": prompt})
    table = pd.DataFrame(rows)
    if len(table) != len(split.candidate_ids) or table.empty:
        raise ValueError("Candidate/prompt lengths disagree.")
    ids = table.global_composition_id.tolist()
    if ids != sorted(set(ids)) or list(zip(table.crop_id, table.disease_id)) != list(split.candidate_pairs):
        raise ValueError("Candidate IDs must be unique ascending global IDs with exact component alignment.")
    if table.prompt.duplicated().any():
        raise ValueError("Distinct candidates normalize to duplicate prompts; review the metadata.")
    return table

prompt_table = build_prompt_table(split)
PROMPT_SHA256 = json_hash({"policy": PROMPT_POLICY, "rows": prompt_table.to_dict("records")})
display(prompt_table)  # All 323 rows remain available; display settings may truncate.
print("Prompt digest:", PROMPT_SHA256)
display(split.fit.disease_name.value_counts().head(12).rename("fitting_count").to_frame())


## 5. Model provenance, native preprocessing and optional acquisition

Use **official OpenAI CLIP**, pinned to commit `d05afc436d78f1c48dc0dbf8e5980a9d471f35f6`. Approved environment additions are CLIP 1.0, ftfy 6.3.1 and regex 2025.9.18; this notebook never installs packages. The pretrained artifacts below are official TorchScript archives with full SHA-256 published in OpenAI's asset URLs. They live outside Git.

Native CLIP preprocessing is shorter-edge bicubic resize to 224, center crop 224, RGB conversion, tensor conversion and CLIP-specific normalization. It is **not WP2's ImageNet pipeline**. Native BPE tokenization uses context length 77 and truncation is forbidden. Long prompts must fail clearly, not silently lose disease tokens.

B/32 is the default resource-efficient reference. L/14 is optional: a successful bounded four-image inference is evidence of feasibility at that batch size, not proof of safe full-run memory or superior plant recognition. There is no automatic model sweep.


In [ ]:
def weight_spec(model_name):
    spec = dict(MODEL_REGISTRY[model_name])
    spec["url"] = f"https://openaipublic.azureedge.net/clip/models/{spec['sha256']}/{spec['filename']}"
    return spec

def verify_weight(path, expected_sha256):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing local CLIP weights: {path}. Explicitly acquire or select the verified file.")
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f"CLIP weight SHA-256 mismatch: {path}")
    return {"path": str(path.resolve()), "sha256": actual, "bytes": path.stat().st_size}

def acquire_official_weight(model_name, directory):
    # Called only by the explicit acquisition guard; never by load_frozen_clip.
    from urllib.request import urlopen
    spec = weight_spec(model_name)
    directory = Path(directory)
    if ROOT.resolve() == directory.resolve() or ROOT.resolve() in directory.resolve().parents:
        raise ValueError("Weights must be stored outside the Git repository.")
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / spec["filename"]
    if target.exists():
        return verify_weight(target, spec["sha256"])
    partial = target.with_suffix(".pt.partial")
    with urlopen(spec["url"], timeout=60) as response, partial.open("xb") as handle:
        for chunk in iter(lambda: response.read(4 * 1024 * 1024), b""):
            handle.write(chunk)
    verify_weight(partial, spec["sha256"])
    # Exclusive destination creation protects against overwriting another process's asset.
    import os
    os.link(partial, target)
    partial.unlink()
    return verify_weight(target, spec["sha256"])

def load_frozen_clip(config):
    import clip
    distribution = metadata.distribution("clip")
    direct = json.loads(distribution.read_text("direct_url.json") or "{}")
    if distribution.version != "1.0" or direct.get("vcs_info", {}).get("commit_id") != CLIP_REVISION:
        raise RuntimeError("Install the approved official CLIP revision; this loader refuses an ambiguous package.")
    spec = weight_spec(config["model"])
    artifact = verify_weight(Path(config["weight_directory"]) / spec["filename"], spec["sha256"])
    device = torch.device(config["device"])
    # A fully hash-verified official local artifact prevents clip.load's named-model downloader.
    model, preprocess = clip.load(artifact["path"], device=device, jit=False)
    model.eval().requires_grad_(False)
    if int(model.visual.input_resolution) != spec["input_resolution"]:
        raise ValueError("Unexpected CLIP image resolution.")
    return model, preprocess, {"backend": "official_openai_clip", "revision": CLIP_REVISION,
        "model": config["model"], "artifact": artifact, "official_url": spec["url"],
        "native_preprocess": repr(preprocess), "context_length": int(model.context_length),
        "tokenizer": "clip.tokenize(context_length=77, truncate=False)",
        "bpe_sha256": sha256_file(Path(clip.__file__).parent / "bpe_simple_vocab_16e6.txt.gz"),
        "parameters": sum(p.numel() for p in model.parameters()),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "logit_scale": model.logit_scale.detach().float().exp().cpu().item(),
        "first_parameter_dtype": str(next(model.parameters()).dtype),
        "image_encoder_dtype": str(getattr(model, "dtype", next(model.parameters()).dtype))}

if ACQUIRE_PRETRAINED_WEIGHTS:
    print(acquire_official_weight(CONFIG["model"], CONFIG["weight_directory"]))
else:
    print("Acquisition skipped: use an already verified local artifact or explicitly enable the guarded cell.")

model = preprocess = MODEL_PROVENANCE = None
if LOAD_CLIP_MODEL:
    model, preprocess, MODEL_PROVENANCE = load_frozen_clip(CONFIG)
    print(json.dumps(MODEL_PROVENANCE, indent=2))
else:
    print("Model loading skipped. No weights are downloaded or loaded by default.")


## 6. Transparent image decoding and DataLoaders

Decode only records passed from the training-derived split. Every filename must belong to the original training CSV; flat filename validation and resolved-path checks prevent path escape. The Dataset returns the **original** crop, disease and composition IDs plus filename. There is no augmentation or resampling for frozen zero-shot evaluation. Both folds use the exact model-native transform returned above.

A file-existence completeness check touches training filenames only; image previews and batch decoding remain explicitly guarded.


In [ ]:
class CLIPDevelopmentDataset(Dataset):
    def __init__(self, frame, image_directory, native_preprocess, allowed_training_names):
        if native_preprocess is None:
            raise ValueError("Supply the selected CLIP model's native preprocessing.")
        if frame.empty or frame.image_name.duplicated().any():
            raise ValueError("A nonempty unique-filename development frame is required.")
        if not set(frame.image_name) <= set(allowed_training_names):
            raise ValueError("Dataset contains filenames outside original training metadata.")
        self.frame = frame.copy().reset_index(drop=True)
        self.image_directory = Path(image_directory).resolve()
        self.preprocess = native_preprocess
        for name in self.frame.image_name:
            if not isinstance(name, str) or name in (".", "..") or any(c in name for c in ("/", "\\", ":")):
                raise ValueError("Only flat training image filenames are allowed.")
            path = (self.image_directory / name).resolve()
            if path.parent != self.image_directory:
                raise ValueError("Image path escapes the approved directory.")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        path = self.image_directory / row.image_name
        with Image.open(path) as image:
            image.load()  # Explicit decoding while the file is open.
            tensor = self.preprocess(image.convert("RGB"))
        if tensor.ndim != 3 or not torch.isfinite(tensor).all():
            raise ValueError(f"Invalid native-preprocessed image: {row.image_name}")
        return tensor, {
            "crop": int(row.crop_id), "disease": int(row.disease_id),
            "composition": int(row.composition_id), "image_name": str(row.image_name),
        }

def make_validation_loader(frame, native_preprocess, config, allowed_training_names):
    dataset = CLIPDevelopmentDataset(frame, ROOT / "DPD_Small_dataset", native_preprocess, allowed_training_names)
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=False, drop_last=False,
        num_workers=config["num_workers"], pin_memory=torch.device(config["device"]).type == "cuda",
        generator=torch.Generator().manual_seed(config["seed"]))

TRAINING_NAMES = set(pd.concat([split.fit, split.val_seen, split.val_unseen]).image_name)
missing_training_files = sorted(name for name in TRAINING_NAMES if not (ROOT / "DPD_Small_dataset" / name).is_file())
print("Training metadata file-existence check:", len(TRAINING_NAMES), "records;", len(missing_training_files), "missing")
if missing_training_files:
    raise FileNotFoundError(f"Missing training images, first examples: {missing_training_files[:5]}")
if SHOW_IMAGES:
    if preprocess is None:
        raise RuntimeError("Enable LOAD_CLIP_MODEL first to inspect native preprocessing.")
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for ax, row in zip(axes, split.fit.head(4).itertuples()):
        with Image.open(ROOT / "DPD_Small_dataset" / row.image_name) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"{row.crop_name}\n{row.disease_name}\nID {row.composition_id}")
        ax.axis("off")
    plt.tight_layout()
else:
    print("Image preview skipped; no images decoded.")


## 7. Native tokenization and frozen text embeddings

The score-column order is the prompt table order. All encoders are frozen and use inference mode. L2 normalization is explicit and zero/non-finite vectors fail. Encoded features are promoted to **CPU float64** for normalization and cosine scoring, giving one documented numerical scoring path for this CLIP reference. This does not change WP2's canonical float64 log-softmax scorer.

CLIP's learned positive logit scale would multiply every cosine score by the same constant; we record it but rank cosine scores directly. No softmax, crop/disease-head formula or candidate-prior calibration is inserted.


In [ ]:
def unit_features(features):
    if not isinstance(features, torch.Tensor) or features.ndim != 2 or not features.is_floating_point():
        raise ValueError("Features must be a floating [N,D] tensor.")
    values = features.detach().to(device="cpu", dtype=torch.float64)
    if values.shape[0] == 0 or values.shape[1] == 0 or not torch.isfinite(values).all():
        raise ValueError("Feature vectors must be nonempty, finite and nonzero.")
    scale = values.abs().amax(dim=1, keepdim=True)
    if (scale == 0).any():
        raise ValueError("Zero feature vectors cannot be normalized.")
    scaled = values / scale
    return scaled / torch.linalg.vector_norm(scaled, dim=1, keepdim=True)

def tokenize_candidates(table):
    import clip
    if table.empty or table.prompt.duplicated().any():
        raise ValueError("Prompts must be nonempty and unique.")
    tokens = clip.tokenize(table.prompt.tolist(), context_length=77, truncate=False)
    if tokens.shape != (len(table), 77):
        raise ValueError("Native tokenizer returned an unexpected shape.")
    return tokens

@torch.inference_mode()
def encode_candidate_texts(model, table, config):
    if model.training or any(p.requires_grad for p in model.parameters()):
        raise ValueError("CLIP must be frozen and in evaluation mode.")
    tokens = tokenize_candidates(table)
    outputs = []
    device = torch.device(config["device"])
    for start in range(0, len(tokens), config["text_batch_size"]):
        outputs.append(model.encode_text(tokens[start:start + config["text_batch_size"]].to(device)).cpu())
    features = unit_features(torch.cat(outputs))
    if features.shape != (len(table), MODEL_REGISTRY[config["model"]]["embedding_dimension"]):
        raise ValueError("Unexpected CLIP text feature contract.")
    return features, {"tokens_shape": list(tokens.shape), "features_shape": list(features.shape),
                      "tokens_sha256": hashlib.sha256(tokens.numpy().tobytes()).hexdigest(),
                      "prompt_sha256": json_hash({"policy": PROMPT_POLICY, "rows": table.to_dict("records")})}

def cosine_candidate_scores(image_features, text_features):
    image = unit_features(image_features)
    text = unit_features(text_features)
    if image.shape[1] != text.shape[1]:
        raise ValueError("Image/text embedding dimensions differ.")
    scores = image @ text.T
    if not torch.isfinite(scores).all():
        raise ValueError("Non-finite cosine similarity.")
    return scores

text_features = TEXT_PROVENANCE = None
if RUN_TEXT_ENCODING:
    if model is None:
        raise RuntimeError("Enable LOAD_CLIP_MODEL before text encoding.")
    text_features, TEXT_PROVENANCE = encode_candidate_texts(model, prompt_table, CONFIG)
    print(TEXT_PROVENANCE)
else:
    print("Tokenization and text encoding skipped.")


## 8. Candidate ranking and evaluation formulas

For each image, rank all candidate cosine similarities. Columns correspond to **ascending original global composition IDs**; a stable descending sort therefore breaks exact score ties by the smaller global ID. Top-k is the fraction whose true global ID is in the first k predictions. No remapping through a dense index is allowed for truth IDs.

For seen accuracy S and internally unseen accuracy U, `H = 2*S*U/(S+U)`, with H=0 when S=U=0. Report H@1 and H@5 separately. These formulas are notebook-local and independently tested against the agreed WP5 interface. There is no change to Jonathan's code or WP2 checkpoint selection.


In [ ]:
def rank_global_ids(scores, candidate_ids):
    if not isinstance(scores, torch.Tensor) or scores.ndim != 2 or not scores.is_floating_point():
        raise ValueError("Scores must be a floating [N,C] tensor.")
    ids = tuple(candidate_ids)
    if not ids or any(type(i) is not int or i < 0 for i in ids) or list(ids) != sorted(set(ids)):
        raise ValueError("Candidates must be unique ascending nonnegative global IDs.")
    if scores.shape[1] != len(ids) or scores.shape[0] == 0 or not torch.isfinite(scores).all():
        raise ValueError("Score shape/values violate the candidate contract.")
    order = torch.argsort(scores.detach().to("cpu", torch.float64), dim=1, descending=True, stable=True)
    return torch.tensor(ids, dtype=torch.int64)[order]

def topk_accuracy(ranked_global_ids, truths, k):
    truths = torch.as_tensor(truths)
    if (ranked_global_ids.ndim != 2 or ranked_global_ids.dtype != torch.int64 or
            truths.ndim != 1 or truths.dtype != torch.int64 or len(truths) != len(ranked_global_ids)
            or len(truths) == 0):
        raise ValueError("Rankings/truths must be nonempty aligned int64 global IDs.")
    if type(k) is not int or not 1 <= k <= ranked_global_ids.shape[1]:
        raise ValueError("k must lie inside the candidate count.")
    if not (ranked_global_ids == truths.cpu()[:, None]).any(dim=1).all():
        raise ValueError("A composition truth is absent from the candidate universe.")
    correct = (ranked_global_ids[:, :k] == truths.cpu()[:, None]).any(dim=1)
    return correct.to(torch.float64).mean().item()

def harmonic_mean(seen, unseen):
    for value in (seen, unseen):
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError("Accuracies must be finite fractions in [0,1].")
    return 0.0 if seen + unseen == 0 else 2.0 * seen * unseen / (seen + unseen)

def fold_metrics(scores, truths, candidate_ids):
    ranked = rank_global_ids(scores, candidate_ids)
    return {"sample_count": len(ranked), "top1": topk_accuracy(ranked, truths, 1),
            "top5": topk_accuracy(ranked, truths, 5)}, ranked

def development_metrics(seen_metrics, unseen_metrics):
    return {"seen_top1": seen_metrics["top1"], "seen_top5": seen_metrics["top5"],
            "unseen_top1": unseen_metrics["top1"], "unseen_top5": unseen_metrics["top5"],
            "h1": harmonic_mean(seen_metrics["top1"], unseen_metrics["top1"]),
            "h5": harmonic_mean(seen_metrics["top5"], unseen_metrics["top5"]),
            "seen_count": seen_metrics["sample_count"], "unseen_count": unseen_metrics["sample_count"]}

# A synthetic exact-tie demonstration, not plant results.
_demo = rank_global_ids(torch.tensor([[1., 1., 0., -1., -2.]], dtype=torch.float64), (2, 7, 19, 23, 90))
assert _demo.tolist() == [[2, 7, 19, 23, 90]]
assert topk_accuracy(_demo, torch.tensor([7]), 1) == 0
assert topk_accuracy(_demo, torch.tensor([7]), 5) == 1
assert harmonic_mean(0., 0.) == 0.
print("Synthetic global-ID tie and metric example passed.")


## 9. Bounded image inference and full-fold inference (separate controls)

The smoke cell decodes at most four **fitting** images, with no optimizer or gradients. It checks dimensions and reports timings/memory; its predictions are not estimates of development accuracy. The full-fold cell below is disabled separately. A shared notebook function handles explicit inference without model-specific assumptions beyond CLIP's public encoding API.

Reported throughput includes DataLoader decoding, native transforms, device transfer and encoding, followed by CPU scoring. Synchronization brackets CUDA timing. Peak allocated and reserved bytes are both recorded; these are not the same as total device usage.


In [ ]:
def verify_evaluation_contract(split, table, text_provenance, model_provenance, config):
    expected = build_prompt_table(split)
    if not table.equals(expected):
        raise ValueError("Candidate/prompt table changed from the pre-specified policy.")
    digest = json_hash({"policy": PROMPT_POLICY, "rows": table.to_dict("records")})
    if text_provenance["prompt_sha256"] != digest:
        raise ValueError("Text embeddings do not match current candidate prompts.")
    if model_provenance["model"] != config["model"] or model_provenance["revision"] != CLIP_REVISION:
        raise ValueError("Loaded CLIP provenance differs from the configured experiment.")
    spec = weight_spec(config["model"])
    if model_provenance["artifact"]["sha256"] != spec["sha256"]:
        raise ValueError("Loaded artifact differs from the pinned model.")
    if sha256_file(split.source_csv) != split.source_sha256:
        raise ValueError("Training metadata changed after split construction.")
    return digest

@torch.inference_mode()
def infer_images(model, loader, candidate_text_features, candidate_ids, device, progress=True):
    from tqdm.auto import tqdm
    if model.training or any(p.requires_grad for p in model.parameters()):
        raise ValueError("Inference requires a frozen eval-mode CLIP.")
    device = torch.device(device)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    scores, truth_parts, crop_parts, disease_parts, filenames = [], [], [], [], []
    for images, targets in tqdm(loader, desc="Frozen CLIP inference", disable=not progress):
        if len(images) == 0 or not torch.isfinite(images).all():
            raise ValueError("Images must be nonempty and finite.")
        image_features = model.encode_image(images.to(device, non_blocking=device.type == "cuda"))
        batch_scores = cosine_candidate_scores(image_features, candidate_text_features)
        if len(batch_scores) != len(images) or batch_scores.shape[1] != len(candidate_ids):
            raise ValueError("Unexpected similarity score shape.")
        scores.append(batch_scores)
        for key, destination in (("composition", truth_parts), ("crop", crop_parts), ("disease", disease_parts)):
            value = torch.as_tensor(targets[key])
            if value.dtype != torch.int64 or value.shape != (len(images),):
                raise ValueError("Targets must retain int64 original global IDs.")
            destination.append(value.cpu())
        filenames.extend(targets["image_name"])
    if not scores:
        raise ValueError("Cannot evaluate an empty loader.")
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - started
    score_tensor, truths = torch.cat(scores), torch.cat(truth_parts)
    if len(set(filenames)) != len(filenames) or len(filenames) != len(truths):
        raise ValueError("Inference contains repeated or misaligned filenames.")
    metrics, ranking = fold_metrics(score_tensor, truths, candidate_ids)
    timing = {"elapsed_seconds": elapsed, "images_per_second": len(truths) / elapsed,
              "peak_allocated_bytes": torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None,
              "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None,
              "score_shape": list(score_tensor.shape),
              "image_embedding_dimension": candidate_text_features.shape[1]}
    return {"metrics": metrics, "ranking": ranking, "scores": score_tensor,
            "truths": truths, "crop_truths": torch.cat(crop_parts), "disease_truths": torch.cat(disease_parts),
            "filenames": filenames, "timing": timing}

smoke_result = None
if RUN_IMAGE_SMOKE:
    if model is None or text_features is None:
        raise RuntimeError("Enable model loading and text encoding first.")
    smoke_frame = split.fit.head(CONFIG["smoke_images"])
    smoke_loader = make_validation_loader(smoke_frame, preprocess, {**CONFIG, "batch_size": len(smoke_frame)}, TRAINING_NAMES)
    smoke_result = infer_images(model, smoke_loader, text_features, split.candidate_ids, CONFIG["device"])
    print("Bounded fitting-image smoke only:", smoke_result["timing"])
else:
    print("Image smoke skipped.")


## 10. Manual development evaluation and results

To evaluate manually, review sections 1–8, enable model loading and text encoding, execute those cells, then explicitly set **RUN_DEVELOPMENT_EVALUATION=True** and execute this cell. It processes exactly 1,238 seen-validation and 1,894 internally unseen-validation images, never official tests. Disable the flag again after use. No trained model selection or fine-tuning occurs here.

Predictions rank the same 323 candidates in both folds. Full-development results remain absent until this cell is actually run.


In [ ]:
seen_result = unseen_result = full_metrics = None
if RUN_DEVELOPMENT_EVALUATION:
    if model is None or text_features is None:
        raise RuntimeError("Load the verified model and encode the fixed candidate prompts first.")
    verify_evaluation_contract(split, prompt_table, TEXT_PROVENANCE, MODEL_PROVENANCE, CONFIG)
    seen_loader = make_validation_loader(split.val_seen, preprocess, CONFIG, TRAINING_NAMES)
    unseen_loader = make_validation_loader(split.val_unseen, preprocess, CONFIG, TRAINING_NAMES)
    seen_result = infer_images(model, seen_loader, text_features, split.candidate_ids, CONFIG["device"])
    unseen_result = infer_images(model, unseen_loader, text_features, split.candidate_ids, CONFIG["device"])
    if (seen_result["metrics"]["sample_count"], unseen_result["metrics"]["sample_count"]) != (1238, 1894):
        raise ValueError("Incomplete development evaluation; do not report partial-fold metrics.")
    full_metrics = development_metrics(seen_result["metrics"], unseen_result["metrics"])
    display(pd.DataFrame([full_metrics]))
    display(pd.DataFrame([{"fold": "seen", **seen_result["timing"]}, {"fold": "internally unseen", **unseen_result["timing"]}]))
else:
    print("Full development inference is disabled. No CLIP development results exist in this session.")


## 11. Per-pair diagnostics, component support and representative errors

These analyses consume predictions already produced above; they do not run another sweep. Per-pair Top-1/Top-5 uses the same global candidate rankings. Component accuracies below refer to the crop/disease of the **top-ranked composition**, not independent crop/disease classifier heads. This distinction prevents a misleading comparison with WP2's component-head diagnostics.

Record fitting support, healthy-prediction frequency and the preference for the 258 fitting compositions. Inspect deterministic first errors and their five ranked label pairs; do not tune baseline prompts from these errors.


In [ ]:
def prediction_diagnostics(result, split, table):
    id_to_pair = dict(zip(table.global_composition_id, zip(table.crop_id, table.disease_id)))
    id_to_text = dict(zip(table.global_composition_id, table.prompt))
    ranked, truths = result["ranking"], result["truths"]
    top1_ids = ranked[:, 0].tolist()
    crop_pred = torch.tensor([id_to_pair[i][0] for i in top1_ids], dtype=torch.int64)
    disease_pred = torch.tensor([id_to_pair[i][1] for i in top1_ids], dtype=torch.int64)
    healthy_ids = {key for key, name in split.mappings["disease_id_to_name"].items() if normalize_label(name) == "healthy"}
    fitting_ids = set(split.fit.composition_id)
    crop_support, disease_support = Counter(split.fit.crop_id), Counter(split.fit.disease_id)
    rows = []
    for global_id in sorted(set(truths.tolist())):
        mask = truths == global_id
        crop, disease = id_to_pair[global_id]
        local_ranks, local_truths = ranked[mask], truths[mask]
        rows.append({"global_composition_id": global_id, "crop_id": crop, "disease_id": disease,
            "crop": split.mappings["crop_id_to_name"][crop], "disease": split.mappings["disease_id_to_name"][disease],
            "sample_count": int(mask.sum()), "top1": topk_accuracy(local_ranks, local_truths, 1),
            "top5": topk_accuracy(local_ranks, local_truths, 5),
            "fitting_crop_support": crop_support[crop], "fitting_disease_support": disease_support[disease],
            "fitting_composition_support": int((split.fit.composition_id == global_id).sum())})
    errors = []
    for index in (ranked[:, 0] != truths).nonzero().flatten().tolist()[:12]:
        top = ranked[index, :5].tolist()
        candidate_columns = {int(global_id): column for column, global_id in enumerate(table.global_composition_id)}
        errors.append({"filename": result["filenames"][index], "truth_global_id": int(truths[index]),
            "truth_prompt": id_to_text[int(truths[index])],
            "top5": [{"global_id": key, "prompt": id_to_text[key],
                      "cosine": result["scores"][index, candidate_columns[key]].item()} for key in top]})
    summary = {
        "top_composition_crop_accuracy": (crop_pred == result["crop_truths"]).double().mean().item(),
        "top_composition_disease_accuracy": (disease_pred == result["disease_truths"]).double().mean().item(),
        "healthy_prediction_frequency": sum(int(i in healthy_ids) for i in disease_pred.tolist()) / len(truths),
        "fitting_composition_prediction_frequency": sum(int(i in fitting_ids) for i in top1_ids) / len(truths),
        "sample_count": len(truths),
    }
    return {"summary": summary, "per_pair": pd.DataFrame(rows), "representative_errors": errors}

seen_diagnostics = unseen_diagnostics = None
if full_metrics is not None:
    seen_diagnostics = prediction_diagnostics(seen_result, split, prompt_table)
    unseen_diagnostics = prediction_diagnostics(unseen_result, split, prompt_table)
    display(pd.DataFrame([{"fold": "seen", **seen_diagnostics["summary"]},
                          {"fold": "internally unseen", **unseen_diagnostics["summary"]}]))
    display(unseen_diagnostics["per_pair"])
    print(json.dumps(unseen_diagnostics["representative_errors"], indent=2))
else:
    print("Diagnostics pending actual owner-executed development inference.")


## 12. Recorded WP2 comparison and per-pair alignment

Read-only comparison uses actual saved WP2 records and verifies source/candidate/split hashes first. The original ConvNeXt-Base selected checkpoint has H@1 ≈4.44%, seen Top-1 ≈49.27%, internally unseen Top-1 ≈2.32%. These are **historical supervised development results**, not CLIP results.

The available per-pair reference is explicitly the **M5E control's selected epoch 1**, which reproduces those aggregate metrics. It is labeled separately and verified against its own run summary; it must not be silently described as a newly re-evaluated historical checkpoint. The capped balancing pilot (H@1 ≈3.11%) is a closed negative result and is not the WP3 comparison target. If evidence paths are absent on another machine, comparison remains pending. No model weights are read here.


In [ ]:
def read_wp2_reference(path, identity, candidate_ids):
    path = Path(path)
    record = json.loads(path.read_text())
    for key in ("source_sha256", "candidate_sha256", "split_sha256"):
        if record["split"].get(key) != identity[key]:
            raise ValueError(f"WP2 and WP3 {key} differ; comparison blocked.")
    if tuple(record["split"]["candidate_ids"]) != tuple(candidate_ids):
        raise ValueError("WP2 candidate ordering differs.")
    selected = record.get("selected")
    if not selected:
        raise ValueError("WP2 run has no selected checkpoint.")
    return record

def read_wp2_pair_reference(diagnostic_path, run_record):
    diagnostic = json.loads(Path(diagnostic_path).read_text())
    fold = diagnostic["folds"]["unseen"]
    metrics = run_record["selected"]["metrics"]
    # The diagnostic filename identifies its epoch; also require exact selected-metric agreement.
    if f"epoch_{int(run_record['selected']['epoch']):04d}_" not in Path(diagnostic_path).name:
        raise ValueError("Diagnostic epoch differs from the selected WP2 reference epoch.")
    if fold["composition_top1"] != metrics["unseen_top1"] or fold["composition_top5"] != metrics["unseen_top5"]:
        raise ValueError("Per-pair diagnostic does not match selected WP2 metrics.")
    pairs = pd.DataFrame(fold["pairs"]).rename(columns={
        "composition_id": "global_composition_id", "top1_accuracy": "wp2_top1", "top5_accuracy": "wp2_top5"})
    if pairs.global_composition_id.duplicated().any() or int(pairs.sample_count.sum()) != fold["sample_count"]:
        raise ValueError("Invalid WP2 per-pair counts.")
    return pairs[["global_composition_id", "sample_count", "wp2_top1", "wp2_top5"]]

wp2_record = wp2_pair_record = wp2_pairs = None
if READ_WP2_RESULTS:
    wp2_record = read_wp2_reference(CONFIG["wp2_run_summary"], SPLIT_IDENTITY, split.candidate_ids)
    wp2_pair_record = read_wp2_reference(CONFIG["wp2_pair_run_summary"], SPLIT_IDENTITY, split.candidate_ids)
    wp2_pairs = read_wp2_pair_reference(CONFIG["wp2_diagnostic_json"], wp2_pair_record)
    rows = [{"approach": "WP2 historical ConvNeXt-Base", **wp2_record["selected"]["metrics"]}]
    if full_metrics is not None:
        rows.append({"approach": "WP3 frozen CLIP " + CONFIG["model"],
                     **full_metrics, "top1_harmonic_mean": full_metrics["h1"], "top5_harmonic_mean": full_metrics["h5"]})
    display(pd.DataFrame(rows))
    if unseen_diagnostics is not None:
        per_pair_comparison = unseen_diagnostics["per_pair"].merge(
            wp2_pairs, on=["global_composition_id", "sample_count"], how="outer", validate="one_to_one", indicator=True)
        if not (per_pair_comparison["_merge"] == "both").all():
            raise ValueError("WP2/WP3 per-pair identities/counts differ.")
        per_pair_comparison["top1_difference"] = per_pair_comparison.top1 - per_pair_comparison.wp2_top1
        print("Per-pair WP2 source: M5E control selected epoch 1, run", wp2_pair_record["run_id"])
        display(per_pair_comparison.drop(columns="_merge"))
else:
    print("Recorded WP2 comparison skipped; no CLIP results are fabricated.")


## 13. Reproducible result export and inference handoff

Frozen CLIP needs no new trained-weight checkpoint: reconstruction uses the exact official local artifact, pinned backend, native preprocessing and recorded candidate prompt table. Store a unique result directory with configuration, code/environment identity, class mappings, all candidate prompts, hashes, metrics, per-pair diagnostics and top-ranked error examples. Do not copy model weights or images into Git.

A future WP4 adapter should accept one RGB PIL image and return ranked global IDs, candidate labels, cosine scores and the normalized image feature, with the same native preprocessing and candidate digest. Training-image retrieval and GUI changes remain WP4 work. WP5 consumes candidate IDs and similarity columns directly; it must not apply the WP2 crop/disease-head scorer to CLIP similarities.


In [ ]:
def code_identity():
    notebook_path = ROOT / "notebooks/wp3_clip_zero_shot.ipynb"
    notebook = json.loads(notebook_path.read_text())
    visible_code = "\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code")
    result = {"notebook_sha256_at_export": sha256_file(notebook_path),
              "visible_code_sha256": hashlib.sha256(visible_code.encode()).hexdigest()}
    for name, args in (("head", ["git", "rev-parse", "HEAD"]), ("branch", ["git", "branch", "--show-current"]),
                       ("status", ["git", "status", "--short"])):
        completed = subprocess.run(args, cwd=ROOT, capture_output=True, text=True, check=False)
        result[name] = completed.stdout.strip() if completed.returncode == 0 else "unavailable"
    return result

def export_results(config, metrics, seen_diag, unseen_diag, model_provenance, text_provenance, split, table, runtime):
    if metrics is None or (metrics["seen_count"], metrics["unseen_count"]) != (1238, 1894):
        raise ValueError("Export requires complete actual development results.")
    prompt_hash = verify_evaluation_contract(split, table, text_provenance, model_provenance, config)
    output_root = Path(config["output_directory"]).resolve()
    if output_root == ROOT.resolve() or ROOT.resolve() in output_root.parents:
        raise ValueError("Experiment outputs must be outside Git.")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + config["model"].replace("/", "-") + "_" + uuid.uuid4().hex[:8]
    directory = output_root / run_id
    directory.mkdir(parents=True, exist_ok=False)
    report = {"run_id": run_id, "approach": "frozen_clip_zero_shot", "configuration": config,
        "environment": ENVIRONMENT, "code": code_identity(), "initialization": model_provenance,
        "runtime": runtime, "text": text_provenance, "prompt_sha256": prompt_hash, "prompt_policy": PROMPT_POLICY,
        "split": split_identity(split), "class_mappings": split.mappings,
        "candidate_ids": list(split.candidate_ids), "candidate_pairs": list(split.candidate_pairs),
        "metrics": metrics, "scoring": "float64_cpu_scaled_unit_feature_cosine_v1",
        "tie_policy": "descending_score_then_ascending_global_composition_id",
        "official_tests_used": False, "training_performed": False,
        "seen_diagnostics": seen_diag["summary"], "unseen_diagnostics": unseen_diag["summary"]}
    with (directory / "run_summary.json").open("x") as handle:
        json.dump(report, handle, indent=2, allow_nan=False)
    table.to_csv(directory / "candidate_prompts.csv", index=False, mode="x")
    for name, diagnostic in (("seen", seen_diag), ("unseen", unseen_diag)):
        diagnostic["per_pair"].to_csv(directory / f"{name}_per_pair.csv", index=False, mode="x")
        with (directory / f"{name}_errors.json").open("x") as handle:
            json.dump(diagnostic["representative_errors"], handle, indent=2, allow_nan=False)
    return directory

if SAVE_RESULTS:
    if full_metrics is None:
        raise RuntimeError("Run complete development inference before exporting actual results.")
    print("Saved:", export_results(CONFIG, full_metrics, seen_diagnostics, unseen_diagnostics,
                                  MODEL_PROVENANCE, TEXT_PROVENANCE, split, prompt_table,
                                  {"seen": seen_result["timing"], "unseen": unseen_result["timing"]}))
else:
    print("Result writing disabled.")


## 14. Visualizations from actual predictions

Do not draw accuracy charts until actual full-fold results exist. Histograms and per-pair charts summarize the evaluated run; they are diagnostics, not additional prompt/model selection runs.


In [ ]:
if full_metrics is not None and unseen_diagnostics is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels = ["Seen Top-1", "Unseen Top-1", "H@1", "Seen Top-5", "Unseen Top-5", "H@5"]
    values = [full_metrics[k] for k in ("seen_top1", "unseen_top1", "h1", "seen_top5", "unseen_top5", "h5")]
    axes[0].bar(labels, values)
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=40)
    axes[0].set_ylabel("Accuracy / harmonic mean (fraction)")
    pair_frame = unseen_diagnostics["per_pair"].sort_values("global_composition_id")
    axes[1].bar(pair_frame.global_composition_id.astype(str), pair_frame.top1)
    axes[1].set_ylabel("Unseen composition Top-1")
    axes[1].set_xlabel("Original global composition ID")
    axes[1].tick_params(axis="x", rotation=90, labelsize=6)
    plt.tight_layout()
else:
    print("Plots pending actual development inference; no placeholder results.")


## 15. Exact manual execution steps and limitations

1. Open this notebook in VS Code **through Ubuntu WSL**, select **as04_yolo_katana**, and execute sections 1–4. Inspect counts, hashes and the entire prompt table.
2. Keep the primary model as `ViT-B/32`. Verify the local official weight path. Acquisition is optional and separately guarded; it never installs packages.
3. Set `LOAD_CLIP_MODEL=True` and rerun section 5. Inspect frozen/trainable parameter counts and native transform. Execute section 6 to define Dataset/loader helpers with image previews off. Set `RUN_TEXT_ENCODING=True` and run section 7, then execute the synthetic metric definitions/example in section 8. Review [323,77] tokens and [323,512] text features.
4. Execute section 9 to define the inference helpers. Leave its smoke disabled or explicitly enable `RUN_IMAGE_SMOKE=True` for four fitting images. A smoke result is not a development result.
5. After owner review, explicitly set `RUN_DEVELOPMENT_EVALUATION=True`, then run section 10 once, followed by diagnostics, optional recorded WP2 comparison and visualizations. Exact fold counts must be 1238 and1894.
6. Set `SAVE_RESULTS=True` only to export the actual completed results into a unique directory outside Git. Turn all execution flags back off after use.
7. Repeat the owner-executed evaluation for `ViT-L/14` in a fresh kernel as a separately recorded run. Change only the model/resource configuration; retain the identical final v2 prompt policy and verify the same printed prompt SHA-256. Rebuild and re-encode all323 v2 texts for each model; never reuse v1 or another model's text embeddings. Native L/14 output dimension is768. Record any different microbatch and memory usage. No automatic model sweep is implemented.

Frozen CLIP may lack precise plant pathology vocabulary and may prefer crop appearance or healthy descriptions. Disease names can be ambiguous, and public pretraining overlap with DPD sources cannot be established. Zero-shot here means no supervised fitting updates; it does not prove absence of web-pretraining overlap. Scores are cosine similarities, **not calibrated disease probabilities**. Native CLIP preprocessing intentionally differs from WP2; shared candidate/split/metric contracts ensure comparability while preprocessing remains appropriate to each pretrained model.

No full development inference, training, official-test inference or claim of a winning CLIP model is performed during WP3A preparation.


## 16. WP3B proposal and oral-defense explanations

**One proposed improvement, not implemented:** a small factorized residual visual adapter with separate crop and disease projections over the frozen CLIP image embedding. Learn only the adapter on fitting images. Score crop and disease component texts separately, then combine those component scores over the unchanged 323 candidate pairs. Avoid learned per-composition lookup parameters. The disease projection is shared across crops and receives component supervision, directly addressing WP2's poor disease transfer. Freeze both CLIP encoders and the fixed text policy initially; do not simultaneously add prompt learning, sampler changes or multiple backbones.

The next plan must specify parameter budget, objective and validation selection before training. Ablate the unchanged direct-text zero-shot reference, the untrained factorized component scorer (to separate factorization from learning), and the one trained adapter under one documented budget. Training uses4868 fitting images only; development metrics select settings, official tests remain frozen. Gains must include unseen disease/per-pair evidence and H@1, not merely fitting loss. Factorization can discard useful crop–disease context, so improvement is a hypothesis.

Five likely questions:

1. **What is zero-shot here?** Frozen web-pretrained CLIP matches images to predefined label descriptions without supervised fitting updates; unseen combinations are absent from fitting, while components and vocabulary come from the training source.
2. **Why native preprocessing/tokenization?** CLIP was trained with its own crop, normalization and BPE vocabulary. WP2's ImageNet transforms would shift its input distribution.
3. **How do predictions preserve IDs?** Every text row retains its original global ID and component pair; score column j maps to that row. Stable descending sorting breaks exact ties by ascending global ID.
4. **Why harmonic mean?** It penalizes a model that performs well on seen compositions but fails on internally unseen ones. H=0 when either accuracy is zero.
5. **Why propose a factorized adapter?** Shared disease supervision across crops may improve disease transfer with few trainable parameters, without memorizing258 fitting-composition IDs. It still requires a controlled ablation and separate approval.

Primary references: [OpenAI CLIP paper](https://proceedings.mlr.press/v139/radford21a.html), [pinned official implementation](https://github.com/openai/CLIP/tree/d05afc436d78f1c48dc0dbf8e5980a9d471f35f6), [WP5 PR #1](https://github.com/FZhen2450/COS30082_ML_Project/pull/1). These support CLIP/protocol context; the proposed factorized adapter is a team research hypothesis, not a claimed reproduced published improvement.
